# Turn the Loop Dial: SenseNova-Loop-9B-Preview (1–4 Passes) vs Plain Qwen3.5-9B

**One plain-English question:** `sensenova/SenseNova-Loop-9B-Preview` takes the 32-layer `Qwen3.5-9B` architecture and runs its **middle 24 layers (`4..27`) twice per token** by default (`4 + 24×2 + 4 = 56` layer executions), stitched together with two-stream Manifold-Constrained Hyper-Connections (`mHC`).

What happens at inference time if you turn that **loop dial** to **1, 2, 3, or 4 passes** (`32, 56, 80, or 104` layer steps per token) without retraining—and is it actually more accurate (or faster per GPU-hour) than just running plain **`Qwen/Qwen3.5-9B`** (`32` layers once)?

### Reproducibility Contract
- **Dataset**: `HuggingFaceH4/MATH-500` (`test` split), fixed `seed = 0`, nested subset `S_5 ⊂ S_50 ⊂ S_200` (`data/subset_200.json`).
- **Decoding**: `enable_thinking=True`, `temperature=1.0`, `top_p=0.95`, `top_k=20`, `min_p=0.0`, `presence_penalty=1.5`, `seed=0`, `1` sample/problem.
- **Hardware**: Recommended Colab GPU with `>= 40 GB` VRAM (`G4` RTX PRO 6000 96 GB or `A100` 80 GB) in `bfloat16` (preserving FP32 `mhc_attn` / `mhc_ffn` routers).

## 1. Clone Repository & Install Pinned Dependencies
We pin `transformers==5.18.0` (matching `SenseNova-Loop-9B-Preview`'s `config.json`) alongside `fla-core` for Triton GatedDeltaNet kernels, without overwriting Colab's CUDA-matched `torch`.

In [ ]:
import os, subprocess, sys
from pathlib import Path

if not Path("/content/loop-dial").exists() and Path("/content").exists():
    subprocess.run(["git", "clone", "https://github.com/DavidFlorez/loop-dial.git", "/content/loop-dial"], check=True)

REPO_ROOT = Path("/content/loop-dial") if Path("/content/loop-dial").exists() else Path.cwd().resolve().parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

import torch
constraint = Path("/tmp/torch_pin.txt")
constraint.write_text(f"torch=={torch.__version__}\n")

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-c", str(constraint),
    "transformers==5.18.0", "accelerate", "safetensors",
    "datasets", "math-verify[antlr4_13_2]", "fla-core", "einops",
    "scipy", "pandas", "matplotlib", "gradio", "-e", "."
], check=True)

subprocess.run(["nvidia-smi"], check=False)
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)

## 2. Verify Fixed `MATH-500` Subset (`seed=0`)
`data/subset_200.json` stores the exact deterministic `N=200` nested problem list (`S_5 ⊂ S_50 ⊂ S_200`) so every condition is evaluated on the exact same math problems.

In [ ]:
from loop_dial.common import load_subset_from_disk
from loop_dial.metrics import DEFAULT_SAMPLING

subset_200 = load_subset_from_disk(n=200)
print(f"Loaded {len(subset_200)} deterministic MATH-500 problems.")
print("First 5 sanity IDs:", [p["unique_id"] for p in subset_200[:5]])
print("Shared sampling config:", DEFAULT_SAMPLING)

## 3. Phase A — Verify the Loop Dial with PyTorch Forward Hooks (`32 / 56 / 80 / 104` Layer Calls)
`SenseNova-Loop-9B-Preview`'s `execution_plan(config)` requires **both** `cfg.loop_settings["eval_count"] = n` and `cfg.loop_settings["counts"] = [n]`, while keeping `mhc_attn` and `mhc_ffn` in `float32`. Below we run `scripts/smoke_test.py` (or load saved `results/smoke.json`) to verify physical decoder-layer calls per generated token over 64 tokens and inspect the 5-problem sanity check.

In [1]:
import json
from loop_dial.common import RESULTS_DIR

RUN_PHASE_A_LIVE = False  # Set True on Colab GPU to re-run Phase A from scratch
if RUN_PHASE_A_LIVE:
    subprocess.run([sys.executable, "scripts/smoke_test.py"], check=True)

smoke_data = json.loads((RESULTS_DIR / "smoke.json").read_text())
print(f"GPU: {smoke_data['gpu_name']} ({smoke_data['gpu_total_gb']} GB)")
print(f"Versions: {smoke_data['versions']}\n")
print("64-token forward-hook decoder-layer verification:")
for row in smoke_data["scaling_64_tokens"]:
    print(
        f"  {row['condition']:10s} | plan_len={row['plan_len']:3d} | "
        f"calls/tok={row['decoder_layer_calls_per_generated_token']:5.1f} | "
        f"{row['ms_per_token']:7.2f} ms/tok ({row['tokens_per_s']:6.2f} tok/s) | "
        f"peak_mem={row['peak_gpu_mem_gb']:.2f} GB"
    )

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition (94.97 GB)
Versions: {'python': '3.13.16', 'torch': '2.11.0+cu130', 'cuda': '13.0', 'transformers': '5.18.0', 'accelerate': '1.15.0', 'safetensors': '0.8.0', 'datasets': '4.8.5', 'math_verify': 'installed', 'fla': '0.6.0', 'causal_conv1d': 'not_installed', 'vllm': 'not_installed'}

64-token forward-hook decoder-layer verification:
  loop_n1    | plan_len= 32 | calls/tok= 32.0 |   95.14 ms/tok ( 10.51 tok/s) | peak_mem=16.76 GB
  loop_n2    | plan_len= 56 | calls/tok= 56.0 |  184.36 ms/tok (  5.42 tok/s) | peak_mem=16.80 GB
  loop_n3    | plan_len= 80 | calls/tok= 80.0 |  270.59 ms/tok (  3.70 tok/s) | peak_mem=16.84 GB
  loop_n4    | plan_len=104 | calls/tok=104.0 |  359.29 ms/tok (  2.78 tok/s) | peak_mem=16.87 GB
  qwen35_9b  | plan_len= 32 | calls/tok= 32.0 |   18.40 ms/tok ( 54.35 tok/s) | peak_mem=33.44 GB


## 4. Phase B — Main `MATH-500` Sweep (`n = 1, 2, 3, 4` + `Qwen/Qwen3.5-9B`)
`scripts/run_sweep.py` is fully resumable: it skips any `unique_id` already saved in `results/<condition>.jsonl`, evaluates nested prefix budgets (`2,048 / 4,096 / 8,192 / 16,384` tokens), and runs a 512-token Pass-2 forced answer (`</think>\nFinal Answer: \boxed{`) whenever a trace hits the cap without emitting a final boxed answer.

In [ ]:
RUN_PHASE_B_LIVE = False  # Set True on Colab GPU to run or resume the sweep
if RUN_PHASE_B_LIVE:
    subprocess.run([
        sys.executable, "scripts/run_sweep.py",
        "--backend", "vllm",  # or "hf" with --batch-size 25
        "--n-problems", "200",
        "--max-new-tokens", "16384",
        "--conditions", "loop_n2", "loop_n1", "loop_n3", "loop_n4", "qwen35_9b",
    ], check=True)

## 5. Phase C — Paired McNemar Tests, Wilson 95% CIs, & Nested Token-Budget Curve
Because every model condition is evaluated on the **exact same problems**, we use two-sided **exact McNemar tests** on discordant pairs (`gained` vs `lost` problems) alongside 95% Wilson confidence intervals.

In [ ]:
import pandas as pd

subprocess.run([sys.executable, "scripts/stats.py"], check=True)
summary_df = pd.read_csv("results/summary.csv")
paired_df = pd.read_csv("results/paired_stats.csv")

display_cols = [
    "label", "layer_executions_per_token", "n_problems",
    "accuracy_strict_pct", "accuracy_forced_pct",
    "ci95_forced_low_pct", "ci95_forced_high_pct",
    "hit_cap_pct", "loop_pct", "median_tokens", "correct_forced_per_gpu_hour"
]
print(summary_df[display_cols].to_string(index=False))
print("\nHeadline Paired Exact McNemar Comparisons (forced-answer metric):")
headline = paired_df[paired_df["metric"] == "forced"]
print(headline[["comparison", "diff_cmp_minus_ref_pp", "gained_by_cmp", "lost_by_cmp", "mcnemar_p_value"]].to_string(index=False))

### Logged `N = 200` Results Snapshot (from `results/summary.csv`)

| Dial setting | Layer steps / token | Solved cleanly (`≤16k` tokens) | With cap forced answer | Hit 16k cap | Median tokens | Strict answers / GPU-hr |
| :--- | ---: | ---: | ---: | ---: | ---: | ---: |
| Loop 1 pass (32 layers) | 32 | **0.5%** (1/200, CI [0.09%, 2.78%]) | **3.5%** (7/200) | 72.5% | 16,384 | 2.1 |
| Loop 2 passes (56 layers, default) | 56 | **99.5%** (199/200, CI [97.22%, 99.91%]) | **99.5%** (199/200) | 0.0% | 572 | 665.8 |
| Loop 3 passes (80 layers) | 80 | **93.5%** (187/200, CI [89.2%, 96.16%]) | **95.5%** (191/200) | 4.5% | 685 | 370.6 |
| Loop 4 passes (104 layers) | 104 | **88.5%** (177/200, CI [83.34%, 92.21%]) | **92.5%** (185/200) | 9.0% | 856 | 228.5 |
| Qwen3.5-9B (32 layers, no loop) | 32 | **85.5%** (171/200, CI [79.95%, 89.71%]) | **96.0%** (192/200) | 14.5% | 5,230 | 581.3 |

#### Paired Exact McNemar Tests (`N = 200` Identical Problems, Two-Sided)

| Comparison | Metric | Accuracy change (pp) | Gained | Lost | Exact McNemar p-value |
| :--- | :--- | ---: | ---: | ---: | ---: |
| Turning dial down: 1 pass vs 2 passes (default) | strict | -99.0 pp | +0 | -198 | `4.98e-60` |
| Turning dial down: 1 pass vs 2 passes (default) | forced | -96.0 pp | +0 | -192 | `3.19e-58` |
| Turning dial up: 3 passes vs 2 passes (default) | strict | -6.0 pp | +1 | -13 | `0.0018` |
| Turning dial up: 3 passes vs 2 passes (default) | forced | -4.0 pp | +1 | -9 | `0.0215` |
| Turning dial up: 4 passes vs 2 passes (default) | strict | -11.0 pp | +0 | -22 | `4.77e-07` |
| Turning dial up: 4 passes vs 2 passes (default) | forced | -7.0 pp | +0 | -14 | `1.22e-04` |
| Default Loop (2 passes) vs plain Qwen3.5-9B | strict | +14.0 pp | +29 | -1 | `5.77e-08` |
| Default Loop (2 passes) vs plain Qwen3.5-9B | forced | +3.5 pp | +8 | -1 | `0.0391` |

## 6. Phase E — Plain-English Charts (`assets/`)
Generate and display the five plain-English charts (verified programmatically to contain zero raw LaTeX tokens or internal config jargon).

In [ ]:
from IPython.display import Image, display

subprocess.run([sys.executable, "scripts/generate_charts.py"], check=True)
for chart_name in [
    "loop_architecture.png",
    "accuracy_vs_passes.png",
    "speed_vs_accuracy.png",
    "budget_curve.png",
    "paired_changes.png",
]:
    print(f"=== assets/{chart_name} ===")
    display(Image(filename=str(REPO_ROOT / "assets" / chart_name), width=900))

## 7. Phase D — Launch the Side-by-Side Gradio Explorer (`app.py`)
Replays all saved runs offline (zero GPU needed), or pass `--enable-live-gpu` on a Colab GPU runtime to also test custom math questions live.

In [ ]:
LAUNCH_GRADIO = False  # Set True to open the interactive side-by-side app
if LAUNCH_GRADIO:
    from app import ReplayStore, build_app, HEAD_META
    demo = build_app(store=ReplayStore(), enable_live_gpu=torch.cuda.is_available())
    demo.launch(head=HEAD_META, share=False)